# Project 01 — Comparing Gradient Descent: Batch GD vs SGD vs Mini-Batch GD

**Ek hi real dataset, ek hi neural network — sirf gradient update ki strategy alag.**

Is project me hum **Forest CoverType dataset** (5,81,012 rows, 54 features, 7 classes) par ek hi neural network ko **3 tareeko** se train karenge:

| # | Method | Ek update me kitna data? |
|---|--------|--------------------------|
| 1 | **Batch Gradient Descent (GD)** | Poora training dataset |
| 2 | **Stochastic Gradient Descent (SGD)** | Sirf 1 sample |
| 3 | **Mini-Batch Gradient Descent** | Chhota batch (64 samples) |

Teeno me **same architecture, same initial weights, same learning rate (0.01), same epochs (10)** — taaki comparison fair ho. Sirf update strategy badlegi.

### Hum kya-kya observe karenge?
1. **Loss vs Epoch** — teeno ka loss kitni speed se girta hai
2. **Loss vs Updates (Noise)** — SGD ka graph noisy kyu hota hai, mini-batch stable kyu
3. **Gradient Noise** — gradient kitna fluctuate karta hai
4. **Loss Landscape / Valley (River Valley Problem)** — teeno minimum tak kis raaste se jaate hain
5. **Accuracy Comparison** — aakhir me jeeta kaun?

### Roadmap
`Step 1` Dataset → `Step 2` Model → `Step 3` Forward + Loss → `Step 4` Backpropagation → `Step 5` Batch GD → `Step 6` SGD → `Step 7` Mini-Batch GD → `Step 8` Visualisations → `Step 9` Final Comparison

## Step 0 — Setup

Libraries import karo aur device select karo. Pehli baar run karne par dataset download hoga (ek baar, phir cache se aayega).

In [ ]:
!pip install torch torchinfo numpy pandas matplotlib scikit-learn

In [ ]:
import copy
import time

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from torch.utils.data import DataLoader, TensorDataset

# Reproducibility - same seed, taaki har run me comparison fair rahe
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Step 1 — Dataset: Download, Explore, Preprocess

**Covertype Dataset** — geographical features (elevation, slope, soil type...) dekh kar predict karna hai ki us area me **kis type ka forest cover** hai.

- **5,81,012 observations** (rows)
- **54 input features**
- **7 classes** (forest cover types, labels 1–7)

In [ ]:
data = fetch_covtype()

X = data.data
# Forest cover type classes are 1-indexed (1 to 7).
# Shift to 0 to 6 for PyTorch CrossEntropyLoss
y = data.target - 1

print("Features:", X.shape)
print("Labels:", y.shape)
print("Unique Classes:", np.unique(y))

In [ ]:
# Train test split (stratify = har class ka same proportion train/test me)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)

### Feature Scaling — StandardScaler

**StandardScaler kya karta hai?** Har feature ko is formula se badalta hai: `scaled = (x - mean) / std` — matlab har feature ka mean **0** aur standard deviation **1** ho jata hai.

**Help kaise karta hai?** Is dataset me kuch features bahut bade hain (Elevation ≈ 3000) aur kuch sirf 0/1 (binary soil features). Bina scaling ke bade features gradient ko dominate karte hain — loss valley bahut lambi aur patli (elongated) ban jaati hai, aur gradient descent zig-zag karta hai. Scaling valley ko zyada round banata hai, training fast aur stable hoti hai.

**Dusri scaling techniques:** `MinMaxScaler` (sab ko 0–1 range me), `RobustScaler` (median/IQR use karta hai, outliers ke liye better), `Normalizer` (har row ko unit length).

> **Note:** `fit_transform` sirf **train** par lagta hai. Test par sirf `transform` — kyunki scaler ko statistics (mean/std) sirf training data se seekhne chahiye. Test par fit karna = **data leakage** (test ki jaankari training me ghus jaati hai) aur mapping bhi inconsistent ho jaati hai.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)   # sirf transform - fit nahi (no data leakage)

print(X_train.shape)
print(y_train.shape)

In [ ]:
# Convert the data into tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)
print("Min label:", y_train_tensor.min().item())
print("Max label:", y_train_tensor.max().item())
print("Unique labels:", torch.unique(y_train_tensor))

## Step 2 — Neural Network Architecture (PyTorch)

**ForestNet** — ek simple MLP (Multi-Layer Perceptron):

`54 features → Linear(128) → ReLU → Linear(64) → ReLU → Linear(7 classes)`

Total parameters: **15,751** (54×128+128 + 128×64+64 + 64×7+7)

> `super().__init__()` kyu? Ye `nn.Module` ki machinery ko activate karta hai — isi se PyTorch ko pata chalta hai ki model ke andar kaun-kaun se layers/parameters hain. Bina iske `model.parameters()` kuch nahi dega aur training impossible ho jayegi.

In [ ]:
class ForestNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(54, 128)
        self.relu1 = nn.ReLU()

        self.layer2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        self.output = nn.Linear(64, 7)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.output(x)
        return x

In [ ]:
model = ForestNet().to(device)

print(model)
print("Device:", device)

In [ ]:
from torchinfo import summary

summary(model, input_size=(32, 54))

## Step 3 — Forward Propagation + Loss (CrossEntropyLoss)

Pehle sirf 5 samples par forward pass karke dekhte hain ki model bina training ke kya output deta hai, aur loss kitna aata hai. Random weights ke saath loss zyada hoga — yahi loss training me kam karna hai.

In [ ]:
# Sample testing - first 5 samples
sample = X_train_tensor[:5].to(device)
print("Sample shape:", sample.shape)

output = model(sample)
print("Output shape:", output.shape)
print(output)

In [ ]:
criterion = nn.CrossEntropyLoss()

sample_labels = y_train_tensor[:5].to(device)
loss = criterion(output, sample_labels)

print("Initial Loss:", loss.item())

## Step 4 — Backpropagation: Gradients Kaise Nikalte Hain

`loss.backward()` chain rule se har parameter ke liye gradient (`param.grad`) calculate karta hai. Gradient ka **norm** batata hai ki us layer ko kitna bada update milega.

Neeche pehle har layer ka gradient norm dekhte hain, phir **ek manual update** karke khud verify karte hain ki ek gradient step se loss sach me kam hota hai ya nahi.

In [ ]:
# Fresh model - one backward pass on 5 samples
model = ForestNet().to(device)
criterion = nn.CrossEntropyLoss()

sample = X_train_tensor[:5].to(device)
sample_labels = y_train_tensor[:5].to(device)

output = model(sample)
loss = criterion(output, sample_labels)
print("Loss:", loss.item())

loss.backward()   # <- backpropagation: saare gradients param.grad me aa gaye

In [ ]:
for name, param in model.named_parameters():
    print("Layer:", name)
    print("  Weight Shape:  ", param.shape)
    print("  Gradient Shape:", param.grad.shape)
    print("  Gradient Norm: ", param.grad.norm().item())

In [ ]:
layers = []
gradient_norms = []

for name, param in model.named_parameters():
    if param.grad is not None:
        layers.append(name)
        gradient_norms.append(param.grad.norm().item())

plt.figure(figsize=(10, 5))
plt.bar(layers, gradient_norms)
plt.xticks(rotation=45)
plt.ylabel("Gradient Norm")
plt.title("Gradient Magnitude Across Neural Network Layers")
plt.tight_layout()
plt.show()

### Ek Manual Weight Update — khud karke dekho

Optimizer jo karta hai, wahi haath se: **θ ← θ − η · ∇θ**

64 samples par forward → loss → backward → manual update → phir se loss. Loss kam hua matlab gradient descent kaam kar raha hai.

In [ ]:
# Fresh model
model = ForestNet().to(device)
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01

sample = X_train_tensor[:64].to(device)
labels = y_train_tensor[:64].to(device)

# Forward + Loss
output = model(sample)
loss = criterion(output, labels)
print("Loss before update:", loss.item())

# Backward
loss.backward()

# Manual weight update: theta = theta - lr * grad
with torch.no_grad():
    for param in model.parameters():
        param -= learning_rate * param.grad
        param.grad.zero_()

# Same sample par phir loss
new_loss = criterion(model(sample), labels)
print("Loss after update: ", new_loss.item())

## Common Settings — Fair Comparison ke Rules

Teeno methods ke liye ye cheezein **bilkul same** hongi:

- Same architecture (ForestNet)
- **Same initial weights** — teeno ek hi starting point se shuru honge
- Same learning rate **η = 0.01**
- Same epochs **10**
- Same loss function (CrossEntropyLoss)

Sirf ek cheez badlegi: **ek update me kitna data use hota hai।**

Teeno methods ka update rule bhi ek hi hai:

- **Batch GD:** θ ← θ − η · ∇J(θ) — poore dataset ka gradient
- **SGD:** θ ← θ − η · ∇Lᵢ(θ) — sirf 1 sample ka gradient
- **Mini-Batch:** θ ← θ − η · (1/64) · Σ ∇Lᵢ(θ) — 64 samples ke gradient ka average

In [ ]:
import copy

# ---- Common settings (teeno methods ke liye same) ----
learning_rate = 0.01
epochs = 10
batch_size_mb = 64        # mini-batch size
sgd_subset_size = 5000    # SGD ke liye fixed subset (neeche note dekho)

criterion = nn.CrossEntropyLoss()

# ---- Same initial weights for all three methods ----
torch.manual_seed(42)
base_model = ForestNet()
initial_weights = copy.deepcopy(base_model.state_dict())

print("Initial weights saved. Teeno methods yahi se start karenge.")

In [ ]:
# ---- Helper functions (teeno training loops inhi ko use karenge) ----

def apply_gradients(model, lr):
    """Ek manual gradient descent update: theta = theta - lr * grad"""
    with torch.no_grad():
        for param in model.parameters():
            if param.grad is not None:
                param -= lr * param.grad

def grad_norm(model):
    """Pehli layer ke weight gradient ka norm - noise measure karne ke liye"""
    return model.layer1.weight.grad.norm().item()

def snapshot(model):
    """Model ke saare weights ek lambe vector me - loss landscape ke liye"""
    return torch.cat([p.detach().cpu().flatten() for p in model.parameters()]).numpy()

def evaluate(model, X, y):
    """Test accuracy (%)"""
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), 8192):
            out = model(X[i:i + 8192])
            correct += (out.argmax(dim=1) == y[i:i + 8192]).sum().item()
    model.train()
    return 100.0 * correct / len(X)

print("Helpers ready.")

## Step 5 — Batch Gradient Descent (Full Batch GD)

**Idea:** Ek update se pehle **poora training dataset** (4,64,809 samples) process hota hai. Gradient pure data ka average hai — isliye sabse **accurate aur smooth**, lekin:

- 1 epoch = sirf **1 update** (10 epochs = sirf 10 updates!)
- Har update mehenga hai (poora data ek saath)

Isliye iska loss curve bilkul smooth hoga, lekin 10 epochs me seekhega kam — updates hi bahut kam hain.

In [ ]:
model_gd = ForestNet().to(device)
model_gd.load_state_dict(copy.deepcopy(initial_weights))   # same start

X_full = X_train_tensor.to(device)
y_full = y_train_tensor.to(device)

loss_history_gd = []   # har epoch ka loss (1 update per epoch)
grad_history_gd = []   # har epoch ka gradient norm
traj_gd = [snapshot(model_gd)]   # loss landscape ke liye weight trajectory

start = time.time()
for epoch in range(epochs):
    model_gd.zero_grad()

    output = model_gd(X_full)          # forward - poora dataset
    loss = criterion(output, y_full)

    loss.backward()                    # backward
    grad_history_gd.append(grad_norm(model_gd))

    apply_gradients(model_gd, learning_rate)   # 1 update

    loss_history_gd.append(loss.item())
    traj_gd.append(snapshot(model_gd))

    print(f"Epoch [{epoch + 1}/{epochs}] Loss: {loss.item():.6f}")

time_gd = time.time() - start
print(f"\nBatch GD done in {time_gd:.1f} sec | Total updates: {epochs} (1 per epoch)")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), loss_history_gd, marker='o', color='#2563EB',
         linewidth=2, label='Batch GD')
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Full Batch Gradient Descent - Loss vs Epoch (bilkul smooth, koi noise nahi)")
plt.legend()
plt.grid(True)
plt.show()

## Step 6 — Stochastic Gradient Descent (SGD)

**Idea:** Har **ek sample** ke baad weights update ho jata hai. Gradient sirf usi ek sample par based hai — isliye kabhi sahi direction, kabhi galat. Yahi **noise** hai.

- Updates bahut zyada hote hain → har epoch me model hazaro baar seekhta hai
- Loss curve jagged/noisy hogi — par trend neeche jayega
- Noise kabhi-kabhi faydemand bhi hai: model sharp/local minima se bahar nikal sakta hai

> **Practical note:** Poore 4,64,809 samples par per-sample loop = 1 epoch me hi 4.6 lakh updates — laptop par ghanto lag jayenge. Isliye hum ek **fixed random subset (5,000 samples)** use kar rahe hain — 10 epochs = 50,000 updates. Noise aur behaviour bilkul waisa hi dikhega, bas run fast hoga. Subset size upar Common Settings me badal sakte ho.

In [ ]:
model_sgd = ForestNet().to(device)
model_sgd.load_state_dict(copy.deepcopy(initial_weights))   # same start

# Fixed random subset for SGD
torch.manual_seed(123)
subset_idx = torch.randperm(len(X_train_tensor))[:sgd_subset_size]
X_sgd = X_train_tensor[subset_idx].to(device)
y_sgd = y_train_tensor[subset_idx].to(device)

print("SGD subset:", X_sgd.shape)

loss_raw_sgd = []     # HAR update ka loss - noise yahi dikhega
loss_epoch_sgd = []   # har epoch ka average loss - GD se compare karne ke liye
grad_raw_sgd = []     # har update ka gradient norm
traj_sgd = [snapshot(model_sgd)]

start = time.time()
for epoch in range(epochs):
    order = torch.randperm(len(X_sgd), device=device)   # har epoch shuffle
    epoch_loss = 0.0

    for idx in order:
        x = X_sgd[idx].unsqueeze(0)   # 1 sample
        y = y_sgd[idx].unsqueeze(0)

        model_sgd.zero_grad()
        output = model_sgd(x)
        loss = criterion(output, y)
        loss.backward()
        grad_raw_sgd.append(grad_norm(model_sgd))
        apply_gradients(model_sgd, learning_rate)   # har sample ke baad update

        loss_raw_sgd.append(loss.item())
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(X_sgd)
    loss_epoch_sgd.append(avg_loss)
    traj_sgd.append(snapshot(model_sgd))

    print(f"Epoch [{epoch + 1}/{epochs}] Avg Loss: {avg_loss:.6f}")

time_sgd = time.time() - start
print(f"\nSGD done in {time_sgd:.1f} sec | Total updates: {len(loss_raw_sgd)}")

In [ ]:
plt.figure(figsize=(12, 5))

# Raw per-update loss - SGD ka asli noisy chehra
plt.plot(loss_raw_sgd, alpha=0.25, color='#DC2626', label="Raw SGD Loss (har update)")

# Moving average - noise ke beech ka trend
window = 100
smooth_loss = np.convolve(loss_raw_sgd, np.ones(window) / window, mode="valid")
plt.plot(range(window - 1, len(loss_raw_sgd)), smooth_loss,
         linewidth=2, color='#DC2626', label="Moving Average (100 updates)")

plt.xlabel("Weight Updates")
plt.ylabel("Loss")
plt.title("Stochastic Gradient Descent - Loss Curve (noisy, par trend neeche)")
plt.legend()
plt.grid(True)
plt.show()

## Step 7 — Mini-Batch Gradient Descent

**Idea:** Beech ka raasta — na poora dataset, na 1 sample. **64 samples** ka batch process karo, unke gradient ka average lo, phir update karo.

- Gradient 64 samples ka average hai → SGD se **bahut kam noise**
- Updates GD se **bahut zyada** (1 epoch ≈ 7,263 updates)
- Matrix operations batch par fast hote hain (GPU/CPU dono ke liye efficient)

Isi balance ki wajah se real duniya me lagbhag har neural network mini-batch se hi train hota hai. Update bhi hum manual hi kar rahe hain — taaki teeno methods me sirf data-feeding ka farak rahe, update rule ka nahi.

In [ ]:
model_mb = ForestNet().to(device)
model_mb.load_state_dict(copy.deepcopy(initial_weights))   # same start

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size_mb, shuffle=True)

print("Batches per epoch:", len(train_loader))

loss_raw_mb = []     # har batch ka loss
loss_epoch_mb = []   # har epoch ka average loss
grad_raw_mb = []     # har batch ka gradient norm
traj_mb = [snapshot(model_mb)]

start = time.time()
for epoch in range(epochs):
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        model_mb.zero_grad()
        output = model_mb(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        grad_raw_mb.append(grad_norm(model_mb))
        apply_gradients(model_mb, learning_rate)   # har batch ke baad update

        loss_raw_mb.append(loss.item())
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    loss_epoch_mb.append(avg_loss)
    traj_mb.append(snapshot(model_mb))

    print(f"Epoch [{epoch + 1}/{epochs}] Avg Loss: {avg_loss:.6f}")

time_mb = time.time() - start
print(f"\nMini-Batch GD done in {time_mb:.1f} sec | Total updates: {len(loss_raw_mb)}")

## Step 8 — Visualisations: Noise, Valley aur Comparison

Ab teeno methods ke recorded data se woh graphs banate hain jinse asal samajh aayegi.

### Graph 1 — Training Loss vs Epochs (teeno ek saath)

Har epoch ke end par teeno kahan khade hain? Dhyaan do: GD ka loss slow girega — galti uski nahi, usko 10 epochs me sirf 10 updates mile. SGD aur Mini-Batch hazaro updates kar chuke honge.

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(range(1, epochs + 1), loss_history_gd, marker='o', color='#2563EB',
         linewidth=2, label='Batch GD (1 update/epoch)')
plt.plot(range(1, epochs + 1), loss_epoch_sgd, marker='s', color='#DC2626',
         linewidth=2, label='SGD (5,000 updates/epoch, subset)')
plt.plot(range(1, epochs + 1), loss_epoch_mb, marker='^', color='#059669',
         linewidth=2, label='Mini-Batch GD (7,263 updates/epoch)')

plt.xlabel("Epoch")
plt.ylabel("Average Training Loss")
plt.title("Graph 1 - Training Loss vs Epochs: GD vs SGD vs Mini-Batch")
plt.legend()
plt.grid(True)
plt.show()

### Graph 2 — Loss vs Updates: Noise Kahan Hai?

Epoch wali nazar se noise chhup jata hai. Ab **har update** ka loss dekhte hain (pehle 2,000 updates):

- **SGD:** har sample alag kahani sunata hai — loss upar-neeche uchhalta rahega (yahi **gradient noise** hai)
- **Mini-Batch:** 64 samples ka average gradient ko shaant kar deta hai — halki si lahar, par stable
- **Batch GD:** is graph me aayega hi nahi — 2,000 updates matlab uske 2,000 epochs!

In [ ]:
N = 2000   # pehle 2000 updates

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# SGD - raw + moving average
axes[0].plot(loss_raw_sgd[:N], alpha=0.3, color='#DC2626', label='Raw loss (har update)')
smooth = np.convolve(loss_raw_sgd[:N], np.ones(50) / 50, mode='valid')
axes[0].plot(range(49, N), smooth, color='#DC2626', linewidth=2, label='Moving average (50)')
axes[0].set_title("SGD - Loss vs Updates (NOISY)")
axes[0].set_xlabel("Weight Updates")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Mini-Batch - raw + moving average
axes[1].plot(loss_raw_mb[:N], alpha=0.3, color='#059669', label='Raw loss (har batch)')
smooth_mb = np.convolve(loss_raw_mb[:N], np.ones(50) / 50, mode='valid')
axes[1].plot(range(49, N), smooth_mb, color='#059669', linewidth=2, label='Moving average (50)')
axes[1].set_title("Mini-Batch GD - Loss vs Updates (STABLE)")
axes[1].set_xlabel("Weight Updates")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Graph 3 — Gradient Noise (Gradient Norm vs Updates)

Noise ko number me bhi dekhte hain: har update par pehli layer ke gradient ka norm.

- SGD ka gradient norm baar-baar uchhalta hai — har sample alag direction kheenchta hai
- Mini-Batch ka norm comparatively shaant band me rehta hai
- Aur neeche printed **standard deviation (std)** batayega ki aakhri epoch me per-update loss kitna fluctuate hua — std jitna bada, noise utna zyada

In [ ]:
N = 1000   # pehle 1000 updates

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(grad_raw_sgd[:N], alpha=0.5, color='#DC2626')
axes[0].set_title("SGD - Gradient Norm vs Updates")
axes[0].set_xlabel("Weight Updates")
axes[0].set_ylabel("Gradient Norm (layer1)")
axes[0].grid(True)

axes[1].plot(grad_raw_mb[:N], alpha=0.5, color='#059669')
axes[1].set_title("Mini-Batch GD - Gradient Norm vs Updates")
axes[1].set_xlabel("Weight Updates")
axes[1].set_ylabel("Gradient Norm (layer1)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Noise in numbers - aakhri epoch ke per-update loss ka standard deviation
sgd_last_epoch = loss_raw_sgd[-sgd_subset_size:]
mb_last_epoch = loss_raw_mb[-len(train_loader):]

print(f"Per-update loss fluctuation (std, last epoch):")
print(f"  SGD:        {np.std(sgd_last_epoch):.4f}   <- sabse zyada noise")
print(f"  Mini-Batch: {np.std(mb_last_epoch):.4f}   <- beech me")
print(f"  Batch GD:   0 updates ke beech fluctuation hi nahi - sirf {epochs} updates total")

### Graph 4 — Loss Landscape: The Valley (River Valley Problem)

**River Valley Problem kya hai?** Real loss surface ek lambi, patli ghaati (ravine) jaisi hoti hai — ek direction me deewarein bahut steep (gradient bada), dusri direction me floor ka slope bahut halka (gradient chhota). Aisi ghaati me:

- **Batch GD** deewaro ke beech zig-zag karta hai — floor ke saath aage badhna slow
- **SGD** noise ki wajah se idhar-udhar uchhalta hai, par floor ke saath drift bhi karta rehta hai
- **Mini-Batch** beech ka raasta — halka zig-zag, steady progress

**Graph kaise bana?** Asli landscape 15,751 dimensions ka hai — dikhaya nahi ja sakta. Isliye hum teeno methods ki **actual weight trajectories** (har epoch ke weights) ko PCA se 2 dimensions me project karke wohi 2D slice dikha rahe hain jisme teeno asal me chale the. Background contours usi slice par full loss ke hain (speed ke liye 20,000 samples par calculate kiye gaye).

Dekho: contours lambe khinche hue (elongated) honge — yahi valley hai. Star = starting point (teeno ka same), X = 10 epochs ke baad ki position.

In [ ]:
# ---- 2D slice banao: teeno trajectories ko PCA se 2D me lao ----
w0 = snapshot(base_model)   # initial weights - teeno yahi se start hue the

all_points = np.array(traj_gd + traj_sgd + traj_mb)   # (33, 15751)
X_centered = all_points - w0

# PCA ke pehle 2 principal components
_, _, Vt = np.linalg.svd(X_centered, full_matrices=False)
pc1, pc2 = Vt[0], Vt[1]

def project(traj):
    pts = (np.array(traj) - w0) @ np.vstack([pc1, pc2]).T
    return pts[:, 0], pts[:, 1]

coords = np.array(all_points - w0) @ np.vstack([pc1, pc2]).T
pad_x = 0.25 * (coords[:, 0].max() - coords[:, 0].min() + 1e-9)
pad_y = 0.25 * (coords[:, 1].max() - coords[:, 1].min() + 1e-9)
a_vals = np.linspace(coords[:, 0].min() - pad_x, coords[:, 0].max() + pad_x, 18)
b_vals = np.linspace(coords[:, 1].min() - pad_y, coords[:, 1].max() + pad_y, 18)

# ---- Har grid point par loss (20,000 samples par - speed ke liye) ----
template = ForestNet().to(device)
param_shapes = [p.shape for p in template.parameters()]
param_sizes = [p.numel() for p in template.parameters()]

def set_weights_from_vector(model, vec):
    with torch.no_grad():
        start = 0
        for p, shape, n in zip(model.parameters(), param_shapes, param_sizes):
            chunk = torch.tensor(vec[start:start + n], dtype=torch.float32).reshape(shape)
            p.copy_(chunk.to(device))
            start += n

X_land = X_train_tensor[:20000].to(device)
y_land = y_train_tensor[:20000].to(device)

Z = np.zeros((len(b_vals), len(a_vals)))
template.eval()
with torch.no_grad():
    for j, b in enumerate(b_vals):
        for i, a in enumerate(a_vals):
            set_weights_from_vector(template, w0 + a * pc1 + b * pc2)
            Z[j, i] = criterion(template(X_land), y_land).item()

print("Loss landscape ready. Min grid loss:", Z.min().round(4))

In [ ]:
plt.figure(figsize=(11, 8))

# Valley ke contours
plt.contourf(a_vals, b_vals, Z, levels=25, cmap='viridis', alpha=0.9)
plt.colorbar(label='Loss')
plt.contour(a_vals, b_vals, Z, levels=12, colors='white', linewidths=0.4, alpha=0.5)

# Teeno ki trajectories
for traj, color, label in [
    (traj_gd, '#2563EB', 'Batch GD'),
    (traj_sgd, '#DC2626', 'SGD'),
    (traj_mb, '#059669', 'Mini-Batch GD'),
]:
    xs, ys = project(traj)
    plt.plot(xs, ys, marker='o', markersize=4, linewidth=1.8, color=color, label=label)
    plt.plot(xs[-1], ys[-1], marker='X', markersize=11, color=color,
             markeredgecolor='white', markeredgewidth=1.2)

# Start point - teeno ka same
plt.plot(0, 0, marker='*', markersize=18, color='yellow',
         markeredgecolor='black', label='Start (same initial weights)')

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Graph 4 - Loss Landscape (Valley) aur Teeno ki Trajectories")
plt.legend(loc='best')
plt.tight_layout()
plt.show()

### Graph 5 — Test Accuracy Comparison

Ab aakhri sawal: 10 epochs ke baad **test data** par kaun sabse sahi predict karta hai?

In [ ]:
X_test_d = X_test_tensor.to(device)
y_test_d = y_test_tensor.to(device)

acc_gd = evaluate(model_gd, X_test_d, y_test_d)
acc_sgd = evaluate(model_sgd, X_test_d, y_test_d)
acc_mb = evaluate(model_mb, X_test_d, y_test_d)

print(f"Batch GD:    {acc_gd:.2f}%")
print(f"SGD:         {acc_sgd:.2f}%")
print(f"Mini-Batch:  {acc_mb:.2f}%")

plt.figure(figsize=(9, 5))
methods = ['Batch GD', 'SGD', 'Mini-Batch GD']
accs = [acc_gd, acc_sgd, acc_mb]
colors = ['#2563EB', '#DC2626', '#059669']

bars = plt.bar(methods, accs, color=colors)
plt.bar_label(bars, fmt='%.2f%%')
plt.ylabel("Test Accuracy (%)")
plt.title("Graph 5 - Test Accuracy Comparison (10 epochs, same lr, same start)")
plt.ylim(0, max(accs) * 1.15)
plt.grid(True, axis='y')
plt.show()

## Step 9 — Final Comparison

Saare numbers ek table me. Table ko padhne ka tareeka:

- **Updates** dekho — 10 epochs sunne me same lagta hai, par asal mehnat updates me hoti hai. GD ne sirf 10 updates kiye, isliye uska loss/accuracy peeche rehna expected hai — ye GD ki haar nahi, **updates ki ginati ka farak** hai. GD ko zyada epochs doge to wo bhi wahi pahunchega, bilkul smooth raaste se.
- **Noise (std)** dekho — SGD sabse noisy, Mini-Batch beech me, GD zero noise.
- **Time** dekho — per-sample loop (SGD) slow hai; batch operations (Mini-Batch) hardware ko khush rakhte hain.

### Conclusion
- **Batch GD** — sabse smooth aur bharosemand gradient, lekin bade data par slow aur kam updates. Chhote datasets ya final fine-tuning ke liye achha.
- **SGD** — sabse tez seekhne ki koshish (har sample par update), par noise bahut. Noise kabhi madad bhi karta hai (bure local minima se escape), par loss curve par bharosa karna mushkil.
- **Mini-Batch GD** — **practical winner**: kam noise, bahut updates, hardware-friendly. Isiliye real duniya ke lagbhag saare frameworks (PyTorch/TensorFlow) default me yahi use karte hain — bas batch size tum choose karte ho.

> **Next steps (chaaho to):** teeno ko same **total updates** par compare karo (GD ko 1000 epochs), learning rate badal kar dekho (0.1 / 0.001), batch size 32/128/512 try karo, aur phir Momentum/Adam jod kar dekho ki valley wala zig-zag kitna kam hota hai.

In [ ]:
results = pd.DataFrame({
    "Method": ["Batch GD", "SGD", "Mini-Batch GD"],
    "Data per Update": ["Full dataset (4,64,809)", "1 sample (5,000 subset)", "64 samples"],
    "Updates per Epoch": [1, sgd_subset_size, len(train_loader)],
    "Total Updates (10 epochs)": [epochs, len(loss_raw_sgd), len(loss_raw_mb)],
    "Final Train Loss": [round(loss_history_gd[-1], 4),
                         round(loss_epoch_sgd[-1], 4),
                         round(loss_epoch_mb[-1], 4)],
    "Test Accuracy (%)": [round(acc_gd, 2), round(acc_sgd, 2), round(acc_mb, 2)],
    "Training Time (sec)": [round(time_gd, 1), round(time_sgd, 1), round(time_mb, 1)],
})

results